### Extra trees:
1) запуск алгоритма с параметрами по умолчанию и вывод некоторой статистики
2) запуск optuna-оптимизации по части гиперпараметров
3) визуализация optuna: важность параметров и контуры
4) запуск алгоритма с найденными гиперпараметрами и вывод предварительной статистики
5) сохраняем результаты дефолтного и оптимизированного алгоритма

In [1]:
from private.utils import get_reduced_mnist_data, memory_check
from public.classification_utils import EXTRA
from public.models import ClassificationProcessor

with memory_check():
    df = get_reduced_mnist_data()
    processor = ClassificationProcessor(df, "label")   
    processor.calculate({
        EXTRA : {},
    })
    processor.report(EXTRA)
    processor.pick_model(EXTRA)

extra_trees took 42.767 seconds

	extra_trees


pr_auc,roc_auc,accuracy
0.994039,0.999163,0.968857


,precision,recall,f1-score,support
0,0.971,0.993,0.982,1381.000
1,0.985,0.985,0.985,1575.000
2,0.966,0.964,0.965,1398.000
3,0.964,0.951,0.957,1428.000
4,0.975,0.966,0.970,1365.000
5,0.967,0.962,0.964,1263.000
6,0.977,0.983,0.980,1375.000
7,0.974,0.968,0.971,1459.000
8,0.960,0.962,0.961,1365.000
9,0.947,0.952,0.949,1391.000


Memory Increased by: 295.19 MB


#### запуск optuna-оптимизации по части гиперпараметров

In [2]:
from public.optuna_utils import OPT_EXTRA, optimize

with memory_check():
    study = optimize(
        model_type=OPT_EXTRA, 
        df=df, 
        target_column="label",
        n_trials=20
    )
    print(f"Наилучшие значения гиперпараметров {study.best_params}")
    print(f"pr_auc на обучающем наборе: {study.best_value:.2f}")

  0%|          | 0/20 [00:00<?, ?it/s]

optuna_optimize took 463.289 seconds
optuna_optimize took 481.992 seconds
optuna_optimize took 495.335 seconds
optuna_optimize took 730.315 seconds
optuna_optimize took 846.428 seconds
optuna_optimize took 1033.151 seconds
optuna_optimize took 208.953 seconds
optuna_optimize took 1196.843 seconds
optuna_optimize took 1901.515 seconds
optuna_optimize took 1448.728 seconds
optuna_optimize took 2196.479 seconds
optuna_optimize took 1390.685 seconds
optuna_optimize took 2459.778 seconds
optuna_optimize took 2174.636 seconds
optuna_optimize took 2345.958 seconds
optuna_optimize took 1881.024 seconds
optuna_optimize took 2985.800 seconds
optuna_optimize took 2522.559 seconds
optuna_optimize took 3087.176 seconds
optuna_optimize took 1965.609 seconds
Наилучшие значения гиперпараметров {'n_estimators': 400, 'max_depth': 26, 'min_samples_leaf': 5, 'criterion': 'gini'}
pr_auc на обучающем наборе: 0.99
Memory Increased by: -477.39 MB


#### Визуализация optuna:
1) Сравнение важности гиперпараметров
2) Отрисовка контура оптимизации. Помогает выбрать направление дальнейшей оптимизации в сторону "темных" областей

In [3]:
from optuna.visualization import plot_param_importances

plot_param_importances(study)

In [4]:
from optuna.visualization import plot_contour

plot_contour(study)

#### Применение найденных лучших гиперпараметров:

In [5]:
with memory_check():
    alter_title = f'{EXTRA}_tuned'
    processor.calculate({
        EXTRA : {
            'n_estimators': 400, 
            'max_depth': 26, 
            'min_samples_leaf': 5, 
            'criterion': 'gini',
            'alter_title': alter_title
        },
    })
    processor.report(alter_title)
    processor.pick_model(alter_title)

extra_trees took 172.405 seconds

	extra_trees_tuned


pr_auc,roc_auc,accuracy
0.991468,0.998710,0.961429


,precision,recall,f1-score,support
0,0.968,0.990,0.979,1381.000
1,0.982,0.982,0.982,1575.000
2,0.958,0.952,0.955,1398.000
3,0.954,0.949,0.951,1428.000
4,0.965,0.959,0.962,1365.000
5,0.966,0.956,0.961,1263.000
6,0.967,0.983,0.975,1375.000
7,0.963,0.963,0.963,1459.000
8,0.955,0.944,0.950,1365.000
9,0.933,0.935,0.934,1391.000


Memory Increased by: -298.24 MB


#### Мини-репорт:

In [6]:
dec_tr = next((model for model in processor.models if model.title == EXTRA), None)
dec_tr_tuned = next((model for model in processor.models if model.title == alter_title), None)

print(f"{EXTRA} : {alter_title} >> {dec_tr.pr_auc} : {dec_tr_tuned.pr_auc}")

extra_trees : extra_trees_tuned >> 0.9940393685389133 : 0.9914678034534014
